## 자치구별 대중교통 스트레스 점수 산출

### 목적
자치구별로 대중교통 이용에서 발생할 수 있는 스트레스 수준을 비교하기 위해,  
버스/지하철 월별 승하차 데이터와 인구 데이터를 결합해 `0~100` 점수를 산출한다.

### 기본 아이디어
스트레스를 한 가지 값으로 단정하지 않고, 아래 3개 관점으로 나눠 본다.

1. **승차 스트레스(발생 압력)**  
   - 지표: `승차인원 / 인구`  
   - 의미: 해당 구 주민이 출발할 때 체감할 수 있는 혼잡 압력

2. **하차 스트레스(유입 압력)**  
   - 지표: `하차인원 / 인구`  
   - 의미: 외부 유입으로 도착지에서 발생할 수 있는 혼잡 압력

3. **방향 불균형 스트레스**  
   - 지표: `|승차 - 하차| / (승차 + 하차)`  
   - 의미: 유출입 흐름이 한쪽으로 쏠릴수록 혼잡/대기 불편이 커질 가능성

### 점수 결합 방식
- 각 지표를 자치구 간 비교 가능하도록 정규화
- 가중치는 두지 않고 단순 평균
- 최종 점수:
  
  `대중교통 스트레스 점수 = (승차점수 + 하차점수 + 불균형점수) / 3`

### 해석 주의
이 점수는 **이용량 기반 상대 지수**이며, 실제 대기시간/배차/혼잡률(차내 밀도)을 직접 측정한 값은 아니다.


In [3]:
# 현재 노트북이 어떤 파이썬 인터프리터에서 실행되는지 확인합니다.
# (환경이 다르면 패키지/결과가 달라질 수 있어 재현성 점검용)
import sys
print(sys.executable)

c:\Users\yiho1\8ssible-Healing-Seoul-Analysis\.venv\Scripts\python.exe


In [4]:
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

In [20]:
import re
import numpy as np
import pandas as pd
from sqlalchemy import create_engine


## 원천 데이터 로드
- `bus_gu_wide`: 자치구별 월별 버스 승하차
- `subway_gu_wide`: 자치구별 월별 지하철 승하차
- `garbage_collection_status`: 자치구별 연도 인구


In [21]:
bus_df = pd.read_sql("SELECT * FROM bus_gu_wide", conn)
subway_df = pd.read_sql("SELECT * FROM subway_gu_wide", conn)

population_raw = pd.read_sql(
    """
    SELECT 자치구,
           `2022_행정구역(A)_인구 (명)`,
           `2023_행정구역(A)_인구 (명)`,
           `2024_행정구역(A)_인구 (명)`
    FROM garbage_collection_status
    """,
    conn
)

print("bus_df:", bus_df.shape)
print("subway_df:", subway_df.shape)
print("population_raw:", population_raw.shape)

display(bus_df.head(2))
display(subway_df.head(2))
display(population_raw.head(2))


bus_df: (25, 74)
subway_df: (25, 76)
population_raw: (25, 4)


,gu_code,gu_name,202201_geton,202201_getoff,202202_geton,202202_getoff,202203_geton,202203_getoff,202204_geton,202204_getoff,...,202408_geton,202408_getoff,202409_geton,202409_getoff,202410_geton,202410_getoff,202411_geton,202411_getoff,202412_geton,202412_getoff
0,11110,종로구,3918759,3744989,3399703,3260335,4117154,3909683,4691873,4452522,...,5115687,4893355,5159753,4912638,5694111,5434703,5518258,5285962,5278899,5076381
1,11140,중구,2807933,2678519,2411271,2295169,2855342,2712931,3343044,3146600,...,3908647,3761632,3844092,3687658,4201329,4031167,4016436,3882968,3925222,3809589


,gu_code,gu_name,202201_geton,202201_getoff,202202_geton,202202_getoff,202203_geton,202203_getoff,202204_geton,202204_getoff,...,202409_geton,202409_getoff,202410_geton,202410_getoff,202411_geton,202411_getoff,202412_geton,202412_getoff,202501_geton,202501_getoff
0,11110,종로구,4342692,4083059,3646743,3436870,4166090,3940136,4768867,4526935,...,5484708,5238690,6427553,6140873,6342368,6024876,6359508,6032470,8371,1829
1,11140,중구,6034978,5843985,5042615,4870897,5760383,5601476,6555818,6420654,...,8353571,8150434,9582007,9343492,9617975,9368612,9655623,9386192,12988,4735


,자치구,2022_행정구역(A)_인구 (명),2023_행정구역(A)_인구 (명),2024_행정구역(A)_인구 (명)
0,종로구,152211,150453,149608
1,중구,130785,131793,131214


## 함수 정의
- 점수는 보기 좋게 `연도별 백분위(rank pct)` 방식으로 0~100 변환
  - 같은 값이 많아도 점수 분포가 더 넓게 보임

In [30]:
def rank_100(s: pd.Series) -> pd.Series:
    # 0~100 백분위 점수(보기 좋은 분포)
    return (s.rank(method="average", pct=True) * 100).astype(float)

def to_yearly_sum_by_gu(df: pd.DataFrame) -> pd.DataFrame:
    # getoff 오타(etoff) 허용
    val_cols = [c for c in df.columns if re.match(r"^\d{6}_(geton|getoff|etoff)$", str(c))]
    if not val_cols:
        raise ValueError("YYYYMM_geton/getoff(etoff) 컬럼을 찾지 못했습니다.")

    tmp = df.copy()
    tmp["gu_name"] = tmp["gu_name"].astype(str).str.strip()

    long_df = tmp[["gu_name"] + val_cols].melt(
        id_vars=["gu_name"], var_name="ym_type", value_name="val"
    )
    long_df["year"] = long_df["ym_type"].str[:4]
    long_df["io"] = long_df["ym_type"].str.split("_").str[-1].str.lower()
    long_df["io"] = long_df["io"].replace({"etoff": "getoff"})  # 오타 보정

    yearly = (
        long_df.groupby(["gu_name", "year", "io"], as_index=False)["val"].sum()
        .pivot(index=["gu_name", "year"], columns="io", values="val")
        .reset_index()
    )

    if "geton" not in yearly.columns:
        yearly["geton"] = 0
    if "getoff" not in yearly.columns:
        yearly["getoff"] = 0

    yearly = yearly.rename(columns={"geton": "year_geton", "getoff": "year_getoff"})
    return yearly

## 인구 데이터 정리 (wide → long)
인구 데이터를 `gu_name, year, population` 형태로 변환합니다.


In [31]:
population_raw.columns = population_raw.columns.astype(str).str.strip()

gu_col = [c for c in population_raw.columns if "자치구" in c][0]
pop_cols = [c for c in population_raw.columns if ("인구" in c and re.search(r"20\d{2}", c))]

population_df = (
    population_raw[[gu_col] + pop_cols]
    .rename(columns={gu_col: "gu_name"})
    .melt(id_vars="gu_name", var_name="pop_col", value_name="population")
)

population_df["year"] = population_df["pop_col"].str.extract(r"(20\d{2})")
population_df["gu_name"] = population_df["gu_name"].astype(str).str.strip()
population_df["population"] = pd.to_numeric(population_df["population"], errors="coerce")
population_df = population_df.dropna(subset=["gu_name", "year", "population"])[["gu_name", "year", "population"]]

print("population_df:", population_df.shape)
display(population_df.head(10))

population_df: (75, 3)


,gu_name,year,population
0,종로구,2022,152211
1,중구,2022,130785
2,용산구,2022,233284
3,성동구,2022,288234
4,광진구,2022,351252
5,동대문구,2022,353601
6,중랑구,2022,390140
7,성북구,2022,441984
8,강북구,2022,297702
9,도봉구,2022,313989


## 교통 데이터 연도 집계
버스/지하철을 각각 연도별 승하차 합계로 변환한 뒤 결합합니다.


In [32]:
bus_y = to_yearly_sum_by_gu(bus_df)
subway_y = to_yearly_sum_by_gu(subway_df)

ys = bus_y.merge(
    subway_y,
    on=["gu_name", "year"],
    how="inner",
    suffixes=("_bus", "_sub")
)

ys["geton_total"] = ys["year_geton_bus"] + ys["year_geton_sub"]
ys["getoff_total"] = ys["year_getoff_bus"] + ys["year_getoff_sub"]

print("교통 연도 데이터:", ys.shape)
print("연도 목록:", sorted(ys["year"].unique()))
display(ys.head(5))


교통 연도 데이터: (75, 8)
연도 목록: ['2022', '2023', '2024']


io,gu_name,year,year_getoff_bus,year_geton_bus,year_getoff_sub,year_geton_sub,geton_total,getoff_total
0,강남구,2022,81542438,84574182,121871497,122171038,206745220,203413935
1,강남구,2023,83608914,86385914,129640432,131263578,217649492,213249346
2,강남구,2024,87233195,90233676,131685645,133232646,223466322,218918840
3,강동구,2022,29741806,31068384,42072612,44446744,75515128,71814418
4,강동구,2023,30994018,32160830,45315675,47997251,80158081,76309693


## 교통+인구 결합
동일한 `gu_name`, `year` 기준으로 병합합니다.


In [33]:
df = ys.merge(population_df, on=["gu_name", "year"], how="inner").copy()
print("결합 데이터:", df.shape)
display(df.head(5))


결합 데이터: (75, 9)


,gu_name,year,year_getoff_bus,year_geton_bus,year_getoff_sub,year_geton_sub,geton_total,getoff_total,population
0,강남구,2022,81542438,84574182,121871497,122171038,206745220,203413935,534103
1,강남구,2023,83608914,86385914,129640432,131263578,217649492,213249346,550282
2,강남구,2024,87233195,90233676,131685645,133232646,223466322,218918840,563215
3,강동구,2022,29741806,31068384,42072612,44446744,75515128,71814418,464037
4,강동구,2023,30994018,32160830,45315675,47997251,80158081,76309693,463318


## 3개 원지표 계산
1. 승차 강도 = `geton_total / population`  
2. 하차 강도 = `getoff_total / population`  
3. 불균형 = `|geton_total - getoff_total| / (geton_total + getoff_total)`

In [34]:
eps = 1e-9
df["board_intensity"] = df["geton_total"] / (df["population"] + eps)
df["alight_intensity"] = df["getoff_total"] / (df["population"] + eps)
df["imbalance"] = (df["geton_total"] - df["getoff_total"]).abs() / (df["geton_total"] + df["getoff_total"] + eps)

display(df[["year", "gu_name", "board_intensity", "alight_intensity", "imbalance"]].head(5))


,year,gu_name,board_intensity,alight_intensity,imbalance
0,2022,강남구,387.088670,380.851512,0.008122
1,2023,강남구,395.523553,387.527388,0.010212
2,2024,강남구,396.769124,388.694974,0.010279
3,2022,강동구,162.735144,154.760112,0.025119
4,2023,강동구,173.008778,164.702630,0.024595


## 점수화 및 최종 점수
- 연도별로 0~100 정규화
- 가중치 없이 3개 점수 단순 평균


In [40]:
df["score_board"] = df.groupby("year")["board_intensity"].transform(rank_100)
df["score_alight"] = df.groupby("year")["alight_intensity"].transform(rank_100)
df["score_imbalance"] = df.groupby("year")["imbalance"].transform(rank_100)

df["transport_stress_score"] = (
    df["score_board"] + df["score_alight"] + df["score_imbalance"]
) / 3

# 보기 좋게 반올림
for c in ["score_board", "score_alight", "score_imbalance", "transport_stress_score"]:
    df[c] = df[c].round(1)

score_by_year = df.sort_values(["year", "transport_stress_score"], ascending=[True, False]).reset_index(drop=True)

display(score_by_year[[
    "year", "gu_name", "population",
    "score_board", "score_alight", "score_imbalance",
    "transport_stress_score"
]])


,year,gu_name,population,score_board,score_alight,score_imbalance,transport_stress_score
0,2022,종로구,152211,96.0,96.0,88.0,93.3
1,2022,마포구,375585,88.0,84.0,92.0,88.0
2,2022,중구,130785,100.0,100.0,56.0,85.3
3,2022,영등포구,398085,64.0,76.0,100.0,80.0
4,2022,서초구,408451,84.0,88.0,60.0,77.3
...,...,...,...,...,...,...,...
70,2024,중랑구,385349,20.0,16.0,60.0,32.0
71,2024,노원구,496552,28.0,32.0,32.0,30.7
72,2024,금천구,239070,24.0,24.0,36.0,28.0
73,2024,도봉구,306032,12.0,8.0,64.0,28.0


In [43]:
for y in sorted(score_by_year["year"].unique()):
    print(f"\n==== {y} ====")
    display(
        score_by_year.loc[score_by_year["year"] == y,
                          ["gu_name","score_board","score_alight","score_imbalance","transport_stress_score"]]
        .sort_values("transport_stress_score", ascending=False)
        .head(25)
    )



==== 2022 ====


,gu_name,score_board,score_alight,score_imbalance,transport_stress_score
0,종로구,96.0,96.0,88.0,93.3
1,마포구,88.0,84.0,92.0,88.0
2,중구,100.0,100.0,56.0,85.3
3,영등포구,64.0,76.0,100.0,80.0
4,서초구,84.0,88.0,60.0,77.3
5,강남구,92.0,92.0,20.0,68.0
6,강북구,56.0,64.0,84.0,68.0
7,용산구,80.0,80.0,40.0,66.7
8,송파구,60.0,56.0,68.0,61.3
9,광진구,68.0,60.0,48.0,58.7



==== 2023 ====


,gu_name,score_board,score_alight,score_imbalance,transport_stress_score
25,종로구,96.0,96.0,80.0,90.7
26,마포구,88.0,88.0,92.0,89.3
27,중구,100.0,100.0,56.0,85.3
28,영등포구,64.0,76.0,100.0,80.0
29,서초구,84.0,84.0,48.0,72.0
30,강북구,56.0,60.0,88.0,68.0
31,강남구,92.0,92.0,16.0,66.7
32,용산구,80.0,80.0,36.0,65.3
33,송파구,60.0,56.0,72.0,62.7
34,광진구,72.0,64.0,44.0,60.0



==== 2024 ====


,gu_name,score_board,score_alight,score_imbalance,transport_stress_score
50,종로구,96.0,96.0,84.0,92.0
51,마포구,88.0,88.0,92.0,89.3
52,중구,100.0,100.0,44.0,81.3
53,영등포구,64.0,76.0,100.0,80.0
54,서초구,80.0,80.0,48.0,69.3
55,강남구,92.0,92.0,16.0,66.7
56,강북구,52.0,60.0,88.0,66.7
57,용산구,84.0,84.0,28.0,65.3
58,광진구,72.0,64.0,56.0,64.0
59,송파구,60.0,56.0,72.0,62.7


In [44]:
score_by_year.to_csv("gu_transport_stress_score_all_years.csv", index=False, encoding="utf-8-sig")
print("저장 완료: gu_transport_stress_score_all_years.csv")

저장 완료: gu_transport_stress_score_all_years.csv
